In [ ]:
pip install lightgbm

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


In [2]:
df = pd.read_csv("/Users/kaviraj/Desktop/GUVI/Project3/regression/r_feature_engineering.csv")

df

,appointment_date_continuous,daily_appointments,age,under_12_years_old,over_60_years_old,SMS_received,Hipertension,Diabetes,Alcoholism,Scholarship,...,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday,lag_1,lag_7,lag_30,rolling_mean_7,rolling_mean_30
0,2020-01-01,25,11.520000,12,0,8,0,0,0,0,...,False,False,False,False,True,0.0,0.0,0.0,0.000000,0.000000
1,2020-01-02,33,16.757576,14,1,6,2,1,0,1,...,False,False,True,False,False,25.0,0.0,0.0,0.000000,0.000000
2,2020-01-03,18,13.388889,9,0,4,0,1,0,2,...,False,False,False,False,False,33.0,0.0,0.0,0.000000,0.000000
3,2020-01-04,16,24.937500,6,3,3,2,1,1,1,...,True,False,False,False,False,18.0,0.0,0.0,0.000000,0.000000
4,2020-01-05,15,16.200000,7,1,2,1,0,0,1,...,False,True,False,False,False,16.0,0.0,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493,2021-05-08,1,9.000000,1,0,0,0,0,0,0,...,True,False,False,False,False,11.0,9.0,3.0,6.142857,3.766667
494,2021-05-09,4,15.500000,1,0,1,0,0,0,0,...,False,True,False,False,False,1.0,9.0,1.0,5.000000,3.700000
495,2021-05-10,3,11.000000,2,0,2,0,0,0,1,...,False,False,False,False,False,4.0,4.0,3.0,4.285714,3.800000
496,2021-05-11,11,17.818182,5,1,5,0,1,0,0,...,False,False,False,True,False,3.0,3.0,3.0,4.142857,3.800000


In [3]:
# convert date from int to datetime
df['appointment_date_continuous'] = pd.to_datetime(df['appointment_date_continuous'])

# int to bool
bool_cols = df.select_dtypes(
    include='bool'
).columns

df[bool_cols] = df[
    bool_cols
].astype(int)

# 1. Train-Test Split (Train: Jan 2020 - Mar 2021, Test: Apr 2021)

In [4]:
# Train Data -> from 2020, Jan to 2021, Feb
# Test data -> 2021, March

train_df = df[
    df['appointment_date_continuous'] < '2021-03-01'
]

test_df = df[
    (df['appointment_date_continuous'] >= '2021-03-01') &
    (df['appointment_date_continuous'] < '2021-04-01')
]

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

Train Shape: (425, 39)
Test Shape: (31, 39)


# 2. Feature & Target

In [6]:
forecast_features = [
    "day",
    "month",
    
    "lag_1",
    "lag_7",
    "lag_30",

    "rolling_mean_7",
    "rolling_mean_30",

    "specialty_psychotherapy",
    "specialty_speech therapy",
    "specialty_physiotherapy",
    "specialty_Unknown",
    "specialty_occupational therapy",

    "day_of_week_Monday",
    "day_of_week_Saturday",
    "day_of_week_Sunday",
    "day_of_week_Thursday",
    "day_of_week_Tuesday",
    "day_of_week_Wednesday"
]

In [7]:
# Create train data
X_train = train_df[forecast_features]

y_train = train_df['daily_appointments']

# Create test data

X_test = test_df[forecast_features]

y_test = test_df['daily_appointments']

# Linear Regression

In [8]:
lr = LinearRegression()

lr.fit(
    X_train,
    y_train
)

y_pred_lr = lr.predict(X_test)
print("Linear Regression")
print("-"*30)

print(
    "MAE:",
    mean_absolute_error(
        y_test,
        y_pred_lr
    )
)

print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(
            y_test,
            y_pred_lr
        )
    )
)

print(
    "R2:",
    r2_score(
        y_test,
        y_pred_lr
    )
)


Linear Regression
------------------------------
MAE: 3.4187786260355835
RMSE: 5.687334565039778
R2: 0.9995506032817082


# XGBoost Regression

In [9]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print("XGBoost")
print("-"*30)

print("MAE :", mean_absolute_error(y_test, y_pred_lr))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("R2 :", r2_score(y_test, y_pred_lr))

XGBoost
------------------------------
MAE : 3.4187786260355835
RMSE : 5.687334565039778
R2 : 0.9995506032817082


# LightGBM

In [10]:
lgbm = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    random_state=42
)

lgbm.fit(X_train, y_train)

y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM")
print("-"*30)

print("MAE :", mean_absolute_error(y_test, y_pred_lr))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("R2 :", r2_score(y_test, y_pred_lr))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000413 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1057
[LightGBM] [Info] Number of data points in the train set: 425, number of used features: 18
[LightGBM] [Info] Start training from score 231.548235
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

# Comparison

In [11]:
results = pd.DataFrame({

'Model':[
    'Linear Regression','XGBoost','LightGBM'],

'MAE':[
mean_absolute_error(y_test,y_pred_lr),
mean_absolute_error(y_test,y_pred_xgb),
mean_absolute_error(y_test,y_pred_lgbm),
],

'RMSE':[
np.sqrt(mean_squared_error(y_test,y_pred_lr)),
np.sqrt(mean_squared_error(y_test,y_pred_xgb)),
np.sqrt(mean_squared_error(y_test,y_pred_lgbm))
],

'R2':[
r2_score(y_test,y_pred_lr),
r2_score(y_test,y_pred_xgb),
r2_score(y_test,y_pred_lgbm)
]
})

results.sort_values(
    by='R2',
    ascending=False
)

,Model,MAE,RMSE,R2
0,Linear Regression,3.418779,5.687335,0.999551
1,XGBoost,7.664793,12.692967,0.997762
2,LightGBM,15.808843,37.090951,0.980886


# Feature Importance

In [12]:
# Linear Regression

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr.coef_
})

importance["Abs"] = abs(
    importance["Coefficient"]
)

importance.sort_values(
    "Abs",
    ascending=False
).head(20)

,Feature,Coefficient,Abs
9,specialty_physiotherapy,1.137112,1.137112
10,specialty_Unknown,1.084256,1.084256
7,specialty_psychotherapy,1.039232,1.039232
8,specialty_speech therapy,1.033119,1.033119
11,specialty_occupational therapy,0.968582,0.968582
16,day_of_week_Tuesday,0.714974,0.714974
15,day_of_week_Thursday,0.713554,0.713554
12,day_of_week_Monday,0.412409,0.412409
14,day_of_week_Sunday,0.327465,0.327465
17,day_of_week_Wednesday,0.317355,0.317355


In [ ]:
# SARIMAX model Train

from statsmodels.tsa.statespace.sarimax import SARIMAX

sarimax_model = SARIMAX(
    y_train,
    order=(1,1,1),
    seasonal_order=(1,1,1,7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_fit = sarimax_model.fit(
    disp=False
)

sarimax_pred = sarimax_fit.forecast(
    steps=len(y_test)
)

print("SARIMAX")
print("-"*30)

print(
    "MAE:",
    mean_absolute_error(
        y_test,
        sarimax_pred
    )
)

print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(
            y_test,
            sarimax_pred
        )
    )
)

print(
    "R2:",
    r2_score(
        y_test,
        sarimax_pred
    )
)

SARIMAX
------------------------------
MAE: 190.04237244059712
RMSE: 280.3377146099523
R2: -0.09188122877449456


In [ ]:
# XGBoost Feature Importance
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

# print(importance)

import matplotlib.pyplot as plt

importance = importance.sort_values(
    by='Importance',
    ascending=True
)

plt.figure(figsize=(10,8))

plt.barh(
    importance['Feature'],
    importance['Importance']
)

plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")

plt.show()

In [ ]:
# LightGBM Feature Importance
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': lgbm.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)
# print(importance)

import matplotlib.pyplot as plt
importance = importance.sort_values(
    by='Importance',
    ascending=True
)

plt.figure(figsize=(10,8))

plt.barh(
    importance['Feature'],
    importance['Importance']
)

plt.title("LightGBM Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Features")

plt.show()